# Lesson 6 — Groupby & aggregation
**Goal:** the single most useful pandas skill — *split, apply, combine*. Answer: which branch, product and channel make the money?

In [1]:
import pandas as pd
orders   = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
products = pd.read_csv('data/products.csv')
orders = orders.merge(products, on=['product_id', 'unit_price'])
orders['revenue'] = orders['quantity'] * orders['unit_price'] * (1 - orders['discount_pct']/100)
orders['profit']  = orders['revenue'] - orders['quantity'] * orders['unit_cost']

## Revenue by branch

In [2]:
orders.groupby('branch')['revenue'].sum().sort_values(ascending=False)

branch
Kariakoo            107063150.0
Mlimani City         98931100.0
Arusha CBD           64620500.0
Mwanza Rock City     49019550.0
Name: revenue, dtype: float64

## Several statistics at once with `agg`

In [3]:
summary = orders.groupby('branch').agg(
    orders=('order_id', 'count'),
    revenue=('revenue', 'sum'),
    avg_order=('revenue', 'mean'),
    profit=('profit', 'sum'),
).sort_values('revenue', ascending=False)
summary['margin_%'] = (summary['profit'] / summary['revenue'] * 100).round(1)
summary.round({'revenue': 0, 'avg_order': 0, 'profit': 0})

,orders,revenue,avg_order,profit,margin_%
branch,,,,,
Kariakoo,835,107063150.0,128219.0,33706150.0,31.5
Mlimani City,720,98931100.0,137404.0,29938100.0,30.3
Arusha CBD,458,64620500.0,141093.0,19845500.0,30.7
Mwanza Rock City,387,49019550.0,126666.0,15477550.0,31.6


## Group by two columns

In [4]:
orders.groupby(['category', 'channel'])['revenue'].sum().unstack().round(0)

channel,In-store,Online
category,,
Accessories,57389550.0,26546600.0
Audio,32562500.0,14456500.0
Networking,12636000.0,4745000.0
Phones,87688250.0,42711500.0
Storage,27535800.0,13362600.0


## `pivot_table` — the Excel pivot, in one line

In [5]:
pd.pivot_table(orders, index='category', columns='branch', values='revenue',
               aggfunc='sum', margins=True, margins_name='TOTAL').round(-3)

branch,Arusha CBD,Kariakoo,Mlimani City,Mwanza Rock City,TOTAL
category,,,,,
Accessories,16266000.0,31173000.0,23530000.0,12966000.0,83936000.0
Audio,7909000.0,14856000.0,16182000.0,8072000.0,47019000.0
Networking,3055000.0,5850000.0,4160000.0,4316000.0,17381000.0
Phones,27287000.0,41018000.0,44752000.0,17342000.0,130400000.0
Storage,10103000.0,14166000.0,10307000.0,6323000.0,40898000.0
TOTAL,64620000.0,107063000.0,98931000.0,49020000.0,319634000.0


## Top products

In [6]:
top = (orders.groupby('product_name')
          .agg(units=('quantity', 'sum'), revenue=('revenue', 'sum'), profit=('profit', 'sum'))
          .sort_values('profit', ascending=False))
top.head(5)

,units,revenue,profit
product_name,,,
Smartphone 128GB,182,112807500.0,18167500.0
Bluetooth Earbuds,337,27523000.0,11347000.0
Wireless Mouse,959,22922500.0,9496500.0
"10,000mAh Power Bank",372,19629500.0,8469500.0
1TB External SSD,113,26088000.0,7443000.0


## `transform` — add a group total back onto every row

In [7]:
orders['branch_revenue'] = orders.groupby('branch')['revenue'].transform('sum')
orders['share_of_branch'] = orders['revenue'] / orders['branch_revenue']
orders[['order_id', 'branch', 'revenue', 'share_of_branch']].head()

,order_id,branch,revenue,share_of_branch
0,O00001,Mlimani City,75000.0,0.000758
1,O00002,Kariakoo,24000.0,0.000224
2,O00003,Mlimani City,23750.0,0.000240
3,O00004,Kariakoo,45000.0,0.000420
4,O00005,Kariakoo,60000.0,0.000560


## Key takeaways
- `df.groupby(keys)[col].agg(...)` — split by keys, summarise each group.
- Named aggregation `agg(new=('col', 'func'))` gives clean column names.
- `pivot_table` for cross-tabs; `transform` to broadcast group results back to rows.

**Exercise:** which payment method has the highest average order value?